In [0]:
# Silver Layer: Sellers Dimension (SCD Type 2 BATCH ITERATOR)

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lower, upper, to_date, to_timestamp,
    when, coalesce, lit, current_timestamp, current_date,
    concat_ws, md5, row_number, desc
)
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE = "ecomstore.ecomlake.bronze_sellers"
SILVER_TABLE = "ecomstore.ecomlake.silver_sellers_scd"
TEMP_CHANGED_TABLE = "ecomstore.ecomlake.temp_changed_sellers"
TEMP_NEW_TABLE ="ecomstore.ecomlake.temp_new_sellers"

bronze_sellers = spark.read.table(BRONZE_TABLE)

# 1. Fetch batches chronologically
batches_df = spark.sql(f"SELECT DISTINCT _batch_id FROM {BRONZE_TABLE} ORDER BY _batch_id").collect()
batches = [row['_batch_id'] for row in batches_df]

for current_batch in batches:
    print(f"\n🚀 Processing Batch: {current_batch}")
    
    incoming_batch = bronze_sellers.filter(col("_batch_id") == current_batch)

    # 2. Deduplicate WITHIN the current batch
    w = Window.partitionBy("seller_id").orderBy(desc("updated_at"), desc("_ingestion_timestamp"))
    
    incoming = (
        incoming_batch
        .withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")
        .withColumn("seller_id",   upper(trim(col("seller_id"))))
        .withColumn("seller_name", trim(col("seller_name")))
        .withColumn("city",        trim(col("city")))
        .withColumn("state",       trim(col("state")))
        .withColumn("seller_tier",
            when(col("seller_tier").isin("Standard", "Premium", "Elite"), col("seller_tier")).otherwise(lit("Standard"))
        )
        .withColumn("is_active",
            when(col("is_active").isin("true", "1", "True"), True)
            .when(col("is_active").isin("false", "0", "False"), False).otherwise(None)
        )
        .withColumn("rating",
            when(col("rating").cast(DoubleType()).between(0, 5), col("rating").cast(DoubleType())).otherwise(None)
        )
        .withColumn("commission_pct", col("commission_pct").cast(DoubleType()))
        .withColumn("total_products", col("total_products").cast("integer"))
        .withColumn("onboarding_date", to_date(col("onboarding_date"), "yyyy-MM-dd"))
        .withColumn("updated_at",      to_timestamp(col("updated_at")))
        .filter(col("seller_id").isNotNull())
        .select(
            "seller_id", "seller_name", "seller_tier", "city", "state",
            "gstin", "rating", "total_products", "is_active",
            "onboarding_date", "commission_pct", "updated_at"
        )
    )

    # 3. Hash the SCD-tracked columns
    scd_cols = ["seller_tier", "city", "commission_pct", "is_active"]
    incoming_with_hash = incoming.withColumn(
        "row_hash",
        md5(concat_ws("||", *[coalesce(col(c).cast("string"), lit("__null__")) for c in scd_cols]))
    )

    # 4. SCD Type 2 Logic
    if spark.catalog.tableExists(SILVER_TABLE):
        existing_scd    = spark.read.table(SILVER_TABLE)
        current_records = existing_scd.filter(col("is_current") == True)

        raw_changed_records = (
            incoming_with_hash.alias("src")
            .join(current_records.select("seller_id", "row_hash").alias("tgt"), on="seller_id", how="inner")
            .filter(col("src.row_hash") != col("tgt.row_hash"))
            .select("src.*")
        )

        raw_new_records = (
            incoming_with_hash.alias("src")
            .join(current_records.select("seller_id").alias("tgt"), on="seller_id", how="left_anti")
        )

        raw_changed_records.coalesce(1).write.format("delta").mode("overwrite").saveAsTable(TEMP_CHANGED_TABLE)
        raw_new_records.coalesce(1).write.format("delta").mode("overwrite").saveAsTable(TEMP_NEW_TABLE)
        
        staged_changed_records = spark.read.table(TEMP_CHANGED_TABLE)
        staged_new_records = spark.read.table(TEMP_NEW_TABLE)

        silver_scd = DeltaTable.forName(spark, SILVER_TABLE)
        (
            silver_scd.alias("target")
            .merge(
                staged_changed_records.alias("source"),
                "target.seller_id = source.seller_id AND target.is_current = true"
            )
            .whenMatchedUpdate(set={
                "is_current": lit(False),
                "end_date":   current_date(),
                "updated_at": current_timestamp()
            })
            .execute()
        )

        def prepare_scd_insert(df):
            return df.select(
                col("seller_id"), col("seller_name"), col("seller_tier"), col("city"),
                col("state"), col("gstin"), col("rating"), col("total_products"),
                col("is_active"), col("onboarding_date"), col("commission_pct"), col("row_hash"),
                current_date().alias("start_date"), lit(None).cast("date").alias("end_date"),
                lit(True).alias("is_current"), current_timestamp().alias("updated_at")
            )

        rows_to_insert = staged_changed_records.unionByName(staged_new_records).transform(prepare_scd_insert)
        rows_to_insert.coalesce(1).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(SILVER_TABLE)

    else:
        initial_scd = incoming_with_hash.select(
            col("seller_id"), col("seller_name"), col("seller_tier"), col("city"),
            col("state"), col("gstin"), col("rating"), col("total_products"),
            col("is_active"), col("onboarding_date"), col("commission_pct"), col("row_hash"),
            current_date().alias("start_date"), lit(None).cast("date").alias("end_date"),
            lit(True).alias("is_current"), current_timestamp().alias("updated_at")
        )
        initial_scd.coalesce(1).write.format("delta").mode("overwrite").option("delta.autoOptimize.optimizeWrite", "true").option("delta.autoOptimize.autoCompact", "true").saveAsTable(SILVER_TABLE)
        print(f"✅ Sellers SCD table created for Batch: {current_batch}")

# Clean up staging tables
spark.sql(f"DROP TABLE IF EXISTS {TEMP_CHANGED_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {TEMP_NEW_TABLE}")